# 🚀 DavidAU Qwen3.8-27B TURBO NEO-CODER-MAX MTP (2x Tesla T4)
## Automated Benchmark Suite (Decode/Output TPS & 100k+ Context Profiler)

### 📌 คำแนะนำการใช้งานบน Kaggle:
1. **Settings ด้านขวา**:
   - **Accelerator**: เลือก `GPU T4 x 2`
   - **Internet**: ปรับเป็น `On`
2. **ขั้นตอนการรัน**:
   - **Cell 1**: ตรวจสอบสถานะ GPU 2x T4 และดาวน์โหลดโมเดล + binary ลง `/kaggle/tmp` (ไม่ชนโควตา 19.5GB ของ `/kaggle/working`)
   - **Cell 2**: รัน **Benchmark Suite** วัด Prefill (pp tok/s), Output Decode (tg tok/s), TTFT และ VRAM ที่ context 512 -> 100k+ tokens
   - **Cell 3**: เริ่มรัน **Production Server (llama-server)** พร้อม Public URL ผ่าน Cloudflare Quick Tunnel
   - **Cell 4**: ทดสอบส่งคำถาม/สั่งเขียนโค้ดผ่าน API ภายใน Notebook

In [ ]:
# ==============================================================================
# 📥 Cell 1: ตรวจสอบ GPU 2x T4 และเตรียมไฟล์โมเดล + llama.cpp CUDA Binaries
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/tmp")
LOG_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ตรวจสอบการ์ดจอผ่าน nvidia-smi
print("🎮 ตรวจสอบสถานะ GPU 2x T4...")
!nvidia-smi --query-gpu=index,name,memory.total,memory.free --format=csv

# ดาวน์โหลดสคริปต์ Benchmark อัตโนมัติ (หากรันบน Kaggle ใหม่)
MODEL_REPO = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF"
MODEL_NAME = "Qwen3.8-27B-TurboFCFusion-735-882-Here-Uncen-NEO-CODER-MAX-MTP-Q4_K_M.gguf"
MODEL_PATH = WORK_DIR / MODEL_NAME
MODEL_URL = f"https://huggingface.co/{MODEL_REPO}/resolve/main/{MODEL_NAME}"

print(f"\n📦 เช็คไฟล์โมเดล: {MODEL_NAME}...")
if not MODEL_PATH.exists():
    print("📥 กำลังดาวน์โหลดโมเดล (~18.5 GB) ไปที่ /kaggle/tmp...")
    !wget --continue --progress=bar:force:noscroll --show-progress "{MODEL_URL}" -O "{str(MODEL_PATH)}"
else:
    print(f"✅ โมเดลพร้อมใช้งาน: {MODEL_PATH} ({MODEL_PATH.stat().st_size / (1024**3):.2f} GB)")


In [ ]:
# ==============================================================================
# 📊 Cell 2: รัน Benchmark Suite (วัด Decode ~TPS, Output ~TPS, Context 100k+)
# ==============================================================================
# รัน Benchmark ทดสอบ Prefill / Decode speed และ VRAM usage ที่แต่ละช่วง context
# context ที่ทดสอบ: 512, 4096 (4k), 16384 (16k), 32768 (32k), 65536 (64k), 102400 (100k+)
!python3 benchmark_qwen38_27b_t4.py \
    --mode all \
    --contexts "512,4096,16384,32768,65536,102400" \
    --gen-tokens 128 \
    --kv-cache q4_0

# แสดงผลสรุป Benchmark เป็น Markdown Table สวยงาม
from IPython.display import Markdown, display
summary_file = Path("/kaggle/working/benchmark_summary_qwen38.md")
if summary_file.exists():
    display(Markdown(summary_file.read_text(encoding="utf-8")))


In [ ]:
# ==============================================================================
# 🚀 Cell 3: เริ่มต้น Production Server (llama-server) + Cloudflare Public URL
# ==============================================================================
import time
import re
import urllib.request

# หยุด service เดิมก่อนเริ่ม
os.system("pkill -9 -f '[l]lama-server' >/dev/null 2>&1")
os.system("pkill -9 -f '[c]loudflared' >/dev/null 2>&1")
time.sleep(1)

# ค้นหา binary
server_bin = str(list(WORK_DIR.rglob("llama-server"))[0])
cf_bin = WORK_DIR / "cloudflared"
if not cf_bin.exists():
    !wget -q --show-progress https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {str(cf_bin)}
    cf_bin.chmod(0o755)

LLAMA_LOG = LOG_DIR / "llama-server.log"
CF_LOG = LOG_DIR / "cloudflared.log"
PORT = 8080

server_cmd = [
    server_bin,
    "-m", str(MODEL_PATH),
    "--alias", "qwen3.8-27b",
    "-ngl", "99",
    "-sm", "layer",
    "-ts", "1,1",
    "-c", "131072",        # 128k context window
    "-np", "1",
    "-fa", "on",
    "-ctk", "q4_0",        # บีบอัด KV cache ป้องกัน OOM ที่ 100k+
    "-ctv", "q4_0",
    "-b", "2048",
    "-ub", "512",
    "--host", "127.0.0.1",
    "--port", str(PORT),
]

print("🚀 Starting llama-server in background...")
log_out = open(LLAMA_LOG, "w", encoding="utf-8")
server_proc = subprocess.Popen(server_cmd, stdout=log_out, stderr=subprocess.STDOUT, text=True, start_new_session=True)

print("⏳ กำลังโหลดน้ำหนักโมเดลเข้าสู่ 2x Tesla T4 VRAM...")
ready = False
t_start = time.time()
while time.time() - t_start < 240:
    if server_proc.poll() is not None:
        print("❌ Server crashed! ดู log:")
        print("\n".join(LLAMA_LOG.read_text().splitlines()[-25:]))
        break
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=3) as resp:
            if resp.status == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if ready:
    print("✅ Local Server READY!")
    # เริ่ม Quick Tunnel
    cf_out = open(CF_LOG, "w", encoding="utf-8")
    cf_proc = subprocess.Popen([str(cf_bin), "tunnel", "--url", f"http://127.0.0.1:{PORT}"], stdout=cf_out, stderr=subprocess.STDOUT, text=True, start_new_session=True)
    
    pub_url = None
    t_cf = time.time()
    while time.time() - t_cf < 30:
        if CF_LOG.exists():
            match = re.search(r"https://[-0-9a-z]+\.trycloudflare\.com", CF_LOG.read_text())
            if match:
                pub_url = match.group(0)
                break
        time.sleep(2)
    
    print("\n" + "=" * 70)
    print("🎉 Qwen3.8-27B MTP SERVER IS ONLINE!")
    print(f"🖥️ Local API : http://127.0.0.1:{PORT}/v1")
    if pub_url:
        print(f"🌐 Public URL: {pub_url}/v1")
    print("=" * 70)


In [ ]:
# ==============================================================================
# 💬 Cell 4: ทดสอบส่งโค้ดและคำสั่งพูดคุย (Test Chat Completion)
# ==============================================================================
import json
import urllib.request

def query_qwen(prompt, max_tokens=512):
    url = 'http://127.0.0.1:8080/v1/chat/completions'
    payload = {
        'model': 'qwen3.8-27b',
        'messages': [
            {'role': 'system', 'content': 'You are an elite software architect. Provide clean, production-grade Python code.'},
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': max_tokens,
        'temperature': 0.7
    }
    req = urllib.request.Request(url, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req, timeout=120) as resp:
        res = json.loads(resp.read().decode('utf-8'))
        return res['choices'][0]['message']['content']

test_prompt = "เขียน Python script Async HTTP Client พร้อม Retry Exponential Backoff และ Connection Pooling"
print(f"👤 คำถาม: {test_prompt}\n")
print("🤖 กำลังสร้างคำตอบ...")
answer = query_qwen(test_prompt, max_tokens=600)
print(f"\n{answer}")


In [ ]:
# ==============================================================================
# 🛑 Cell 5: สั่งหยุดการทำงานของ Server ทั้งหมด
# ==============================================================================
os.system("pkill -9 -f '[l]lama-server' >/dev/null 2>&1")
os.system("pkill -9 -f '[c]loudflared' >/dev/null 2>&1")
print("✅ ปิด Service ทั้งหมดเรียบร้อยแล้ว")
